# Digitális aláírások

**Cél**: a szimmetrikus MAC taghez hasonlóan egy autentikációs üzenet gyártása:
- az aláíró (signer) létrehoz egy nyilvános és titkos kulcsot
- az aláíró aláírja az üzenetet a titkos kulccsal
- bárki (verifier) verifikálja az aláírás helyességét a nyilvános kulccsal

## MAC kontra digitális aláírás

Közös cél: az üzenet integritásának védelme

MAC: mindenkivel, akivel üzenetet cserélünk, generálunk egy titkos kulcsot és azzal verifikálunk

Digitális aláírás: egy nyilvános *pk* kulcsot osztunk meg mindenkivel és azzal verifikálunk
- elég egy aláírást generálni, amit bárki verifikálhat (*publicly verifiable*)
- egy $\sigma$ aláírást egy $m$ üzenethet tovább lehet adni egy harmadik személynek (*transferable*), aki verifikálhatja
- nem letagadható, hogy valaki aláírt egy üzenetet (*non-repudiation*)

Hátrány: az aláírások nagyobbak, mint a MAC tag-ek

## Digital Signature Algorithm (1991-2024)

DSA algoritmus: [NIST FIPS 186-4](https://csrc.nist.gov/publications/detail/fips/186/4/final)

1. Kulcsgenerálás:
    - Válasszunk $p,q$ prímszámokat úgy, hogy $p \equiv 1 \pmod{q}$. Legyen $L,N$ a két prímszám hossza bitekben. Javasolt választások: $(1024, 160), (2048, 224), (2048, 256), (3072, 256)$
    - Legyen $h \in_R \{2,\dots,p-2\}$
    - Legyen $g = h^{(p-1)/q} \pmod{p}$ (de más módszer is létezik, lásd algoritmus)
    - A $(p,q,g)$ hármas lehet nyilvános információ
    - Legyen $x \in_R \{1,2,\dots,q-1\}$ és $y = g^x \pmod{p}$
    - Az $x$ a titkos kulcs, $y$ a nyilvános kulcs
2. Aláírás
    - Az aláíró választ egy hash függvényt. Ha a hash kimenete nagyobb, mint $N = |q|$, a hash kimenetének csak az első $N$ bitjét használjuk fel.
    - Az aláíró választ egy tetszőleges $k \in_R \{1,2,\dots,q-1\}$
    - Legyen $r = (g^k \mod{p}) \mod{q}$
    - Legyen $s = k^{-1} (H(m) + xr) \mod{q}$
    - Az aláírás az $(r, s)$ pár lesz.
3. Verifikálás
    - Legyen $w = s^{-1} \mod{q}$
    - Legyen $u_1 = H(m)w \mod{q}$
    - Legyen $u_2 = rw \mod{q}$
    - Legyen $v = (g^{u_1}y^{u_2} \mod{p}) \mod{q}$
    - Az aláírás helyes, ha $v = r$
    
**Feladat**: Mutassuk meg, hogy a fenti algoritmus helyes!

**Figyelem**: A $k$ értékének megválasztásakor nagyon kell figyelni, hogy azt véletlenszerűen válasszuk! 
- OpenSSL, 2008: a PRNG-ben kódoptimalizálás miatt (Valgrind) maximum 32768 különböző seed értéket lehetett adni a PRNG-nek.
- Playstation 3, 2010: Ugyanaz a $k$ több aláíráshoz $\rightarrow$ titkos kulcs megszerzése

## Hogyan titkosítsunk és írjunk alá `GnuPG`-vel?

[`GnuPG`](https://gnupg.org/) az OpenPGP ([RFC4880](https://www.ietf.org/rfc/rfc4880.txt)) open-source implementációja

Nyilvános kulcs generálás:
- `gpg --full-generate-key`
- `gpg --list-secret-keys`
  ```
  sec   rsa2048/1A4FCCB2EA9CAE58 2023-05-24 [SC]
      3126E047F00A2FF5B6B866EE1A4FCCB2EA9CAE58
  uid                 [ultimate] Hanyecz Ottó (ELTE Kripto gyakorlat) <ohanyecz@inf.elte.hu>
  ssb   rsa2048/B3801B4006BBB807 2023-05-24 [E]
  ```
- `gpg --armor --export 3126E047F00A2FF5B6B866EE1A4FCCB2EA9CAE58 > pubkey.txt`

Egy fájl titkosítása és visszafejtése:
- `gpg --recipient 3126E047F00A2FF5B6B866EE1A4FCCB2EA9CAE58 --encrypt test.txt`
- `gpg --decrypt test.txt.gpg`

Egy fájl aláírása:
- `gpg --local-user 3126E047F00A2FF5B6B866EE1A4FCCB2EA9CAE58 --clearsign test.txt`
- `gpg --verify test.txt.gpg`

Értelemszerűen a kettő kombinálható. Ilyenkor a visszafejtés során a `gpg` ellenőrzi az aláírást is (NE használjuk ugyanazt a kulcsot titkosításhoz és aláíráshoz):
- `gpg --local-user 3126E047F00A2FF5B6B866EE1A4FCCB2EA9CAE58 --recipient 3126E047F00A2FF5B6B866EE1A4FCCB2EA9CAE58 --armor --sign --encrypt test.txt`
- `gpg --decrypt test.txt.asc`
```
gpg: encrypted with 2048-bit RSA key, ID B3801B4006BBB807, created 2023-05-24
      "Hanyecz Ottó (ELTE Kripto gyakorlat) <ohanyecz@inf.elte.hu>"
Ez egy teszt file.
gpg: Signature made 2023. máj. 24., szerda, 10:10:14 CEST
gpg:                using RSA key 3126E047F00A2FF5B6B866EE1A4FCCB2EA9CAE58
gpg: Good signature from "Hanyecz Ottó (ELTE Kripto gyakorlat) <ohanyecz@inf.elte.hu>" [ultimate]
```

Hogyan lehet hozzáadni a nyilvános kulcsot?
- `gpg --import someones_pubkey.txt`
- `gpg --edit-key <név>` futtatásával beállítjuk a trust level-t egy interaktív prompton